# Decision Tree Regression — Dallas 311 Code Compliance

**Standalone notebook** — no repository imports needed. Just point `DATA_PATH` at your CSV.

### What this notebook does
Trains four tree-based regressors to predict `hours_to_close` (how long a Code Compliance
service request takes to be resolved) and walks through every result in detail.

| Model | Type | Key property |
|---|---|---|
| Decision Tree | Single tree | Fully interpretable, fast, prone to overfit |
| Random Forest | Bagged ensemble | Lower variance than a single tree |
| Gradient Boosting | Boosted ensemble | Fits residuals sequentially, often highest accuracy |
| XGBoost | Boosted ensemble | Same idea as GBR, faster, regularised |

### A note on R² before you start
This dataset has a **skewness of ~6.4** and a max value of 31,000h.
A small number of very slow requests dominate the variance, making R² naturally low
for any model. That is a property of the data, not a bug. The more useful metrics
here are **MAE** (typical error per prediction) and the **error-by-bucket** chart
which shows where each model struggles.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.preprocessing  import OrdinalEncoder, OneHotEncoder
from sklearn.impute          import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.tree            import DecisionTreeRegressor, plot_tree, export_text
from sklearn.ensemble        import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics         import mean_squared_error, mean_absolute_error, r2_score

# XGBoost is optional — falls back gracefully if not installed
try:
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    print('XGBoost not installed — will run 3 models instead of 4.')
    print('Install with: pip install xgboost')

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)
plt.rcParams.update({'figure.dpi': 130,
                     'axes.spines.top': False,
                     'axes.spines.right': False})

C_BLUE, C_TEAL, C_RED, C_AMBER, C_GRAY = (
    '#378ADD', '#1D9E75', '#E24B4A', '#BA7517', '#888780'
)
MODEL_COLORS = {
    'Decision Tree':     C_GRAY,
    'Random Forest':     C_TEAL,
    'Gradient Boosting': C_AMBER,
    'XGBoost':           C_BLUE,
}

# ── Update this path to point at your CSV ────────────────────────────────
DATA_PATH    = 'sampled_data.csv'
RANDOM_STATE = 42

# ── Stratified sampling config ───────────────────────────────────────
# Set SAMPLE_FRAC to 1.0 to use the full file, or e.g. 0.5 for half.
# Sampling is stratified by Department so every department keeps the
# same proportional representation regardless of sample size.
# MIN_DEPT_COUNT: departments below this row count are grouped as 'Other'
# before sampling so tiny departments don't produce empty strata.
SAMPLE_FRAC     = 1.0
MIN_DEPT_COUNT  = 50

print('Imports OK.')

In [ ]:
# ── 1. Load raw CSV ───────────────────────────────────────────────────────
df_raw = pd.read_csv(DATA_PATH)
print(f'Loaded {len(df_raw):,} rows × {df_raw.shape[1]} columns')

# ── Stratified sample by Department ──────────────────────────────────
if SAMPLE_FRAC < 1.0:
    # Collapse rare departments into 'Other' so every stratum has
    # enough rows to sample from.
    dept_counts = df_raw['Department'].value_counts()
    common_depts = dept_counts[dept_counts >= MIN_DEPT_COUNT].index
    df_raw['_dept_strata'] = df_raw['Department'].where(
        df_raw['Department'].isin(common_depts), 'Other'
    )
    df_raw = (
        df_raw
        .groupby('_dept_strata', group_keys=False)
        .sample(frac=SAMPLE_FRAC, random_state=RANDOM_STATE)
        .drop(columns=['_dept_strata'])
        .reset_index(drop=True)
    )
    print(f'After stratified sample (frac={SAMPLE_FRAC}): {len(df_raw):,} rows')

# Show department breakdown so you can verify proportions
print('\nDepartment breakdown:')
print(df_raw['Department'].value_counts().to_string())

# ── 2. Parse dates & compute target (hours to close) ─────────────────────
DATE_FMT = '%Y %b %d %I:%M:%S %p'
df = df_raw.copy()
df['Created Date'] = pd.to_datetime(df['Created Date'], format=DATE_FMT, errors='coerce')
df['Closed Date']  = pd.to_datetime(df['Closed Date'],  format=DATE_FMT, errors='coerce')
df['hours_to_close'] = (
    (df['Closed Date'] - df['Created Date']).dt.total_seconds() / 3600
)
# Keep only rows with a valid, non-negative close time
df = df[df['hours_to_close'] >= 0].copy()
print(f'Rows with valid close time: {len(df):,}')

# ── 3. Feature engineering ────────────────────────────────────────────────
# Extract numeric ERT (e.g. '4 Business Days' → 4.0)
df['ERT_days'] = (
    df['ERT (Estimated Response Time)']
    .str.extract(r'(\d+)')[0]
    .astype(float)
)
# Calendar features from Created Date
df['month']       = df['Created Date'].dt.month
df['day_of_week'] = df['Created Date'].dt.dayofweek   # 0 = Monday
df['hour']        = df['Created Date'].dt.hour

# ── 4. Drop leakage + irrelevant columns ─────────────────────────────────
# 'Overall Service Request Due Date' has ~1,200 near-unique values — OHE would
# create 1,200 useless columns.  Drop it before encoding.
# Group rare departments into 'Other' before dropping raw Department
dept_counts  = df['Department'].value_counts()
common_depts = dept_counts[dept_counts >= MIN_DEPT_COUNT].index
df['Department_grouped'] = df['Department'].where(
    df['Department'].isin(common_depts), 'Other'
)

COLS_TO_DROP = [
    'Service Request Number', 'Unique Key', 'Address', 'Department',
    'Closed Date',                       # leakage
    'Update Date',                       # leakage
    'Status',                            # leakage
    'Outcome',                           # leakage
    'Lat_Long Location',                 # high-precision, not useful
    'Overall Service Request Due Date',  # near-unique timestamps
    'Created Date',                      # already extracted
    'ERT (Estimated Response Time)',     # replaced by ERT_days
]
df = df.drop(columns=[c for c in COLS_TO_DROP if c in df.columns])

print(f'Columns after drop: {list(df.columns)}')
print(f'Missing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}')

In [ ]:
# ── Fix City Council District ─────────────────────────────────────────────
# On the full multi-department dataset this column can contain values like
# '7,9' (a request spanning two districts). sklearn cannot fit on strings,
# so we take the first district number when multiple are listed.
# Convert to numeric first, then fill NaN from the same numeric series
ccd = pd.to_numeric(
    df['City Council District'].astype(str).str.split(',').str[0].str.strip(),
    errors='coerce'
)
df['City Council District'] = ccd.fillna(ccd.median())

# ── 5. Encode categorical features ───────────────────────────────────────
# Collapse rare Service Request Types to 'Other'
TOP_N_SRT = 15
top_types = df['Service Request Type'].value_counts().nlargest(TOP_N_SRT).index
df['Service Request Type'] = df['Service Request Type'].where(
    df['Service Request Type'].isin(top_types), 'Other'
)
print('Service Request Type distribution after grouping:')
print(df['Service Request Type'].value_counts())

# Ordinal-encode low-cardinality categoricals
# Include Department_grouped so the model knows which department
# handled each request — a strong predictor of resolution time.
ORD_COLS = ['Priority', 'Method Received Description', 'Department_grouped']
# Only encode Department_grouped if it exists (multi-dept datasets)
ORD_COLS = [c for c in ORD_COLS if c in df.columns]
oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
df[ORD_COLS] = oe.fit_transform(df[ORD_COLS].astype(str))

# One-hot-encode Service Request Type
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore', drop='first')
srt_enc  = ohe.fit_transform(df[['Service Request Type']])
srt_cols = ohe.get_feature_names_out()
df_srt   = pd.DataFrame(srt_enc, columns=srt_cols, index=df.index)
df = pd.concat([df.drop(columns=['Service Request Type']), df_srt], axis=1)

# Impute ALL numeric columns with their median.
# GradientBoostingRegressor (unlike RF/DT) raises on any NaN,
# so we sweep the full matrix rather than just ERT_days.
num_cols = df.select_dtypes(include=['number']).columns.difference(['hours_to_close'])
imp = SimpleImputer(strategy='median')
df[num_cols] = imp.fit_transform(df[num_cols])

print(f'\nFinal feature matrix: {df.shape[0]:,} rows × {df.shape[1] - 1} features')
print(f'Features: {[c for c in df.columns if c != "hours_to_close"]}')

In [ ]:
# ── 6. Train / test split ─────────────────────────────────────────────────
X = df.drop(columns=['hours_to_close'])
y = df['hours_to_close']
FEATURE_NAMES = list(X.columns)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print(f'Train: {len(X_train):,} rows    Test: {len(X_test):,} rows')
print(f'\nTarget (hours_to_close) — train set:')
print(y_train.describe().round(1))
print(f'\nSkewness: {y_train.skew():.2f}  (> 1 = heavily right-skewed)')

In [ ]:
# ── 7. Train all models ───────────────────────────────────────────────────
print('Training models...\n')

dt = DecisionTreeRegressor(
    max_depth=10, min_samples_leaf=5, random_state=RANDOM_STATE
)
dt.fit(X_train, y_train)
print('  Decision Tree        done')

rf = RandomForestRegressor(
    n_estimators=200, max_depth=10, min_samples_leaf=5,
    random_state=RANDOM_STATE, n_jobs=-1
)
rf.fit(X_train, y_train)
print('  Random Forest        done')

gbr = GradientBoostingRegressor(
    n_estimators=200, max_depth=5, learning_rate=0.05,
    subsample=0.8, random_state=RANDOM_STATE
)
gbr.fit(X_train, y_train)
print('  Gradient Boosting    done')

models = {
    'Decision Tree':     dt,
    'Random Forest':     rf,
    'Gradient Boosting': gbr,
}

if XGBOOST_AVAILABLE:
    xgb = XGBRegressor(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        random_state=RANDOM_STATE, eval_metric='rmse',
        n_jobs=-1, verbosity=0
    )
    xgb.fit(X_train, y_train)
    models['XGBoost'] = xgb
    print('  XGBoost              done')

print(f'\nAll {len(models)} models trained.')

In [ ]:
# ── 8. Evaluate ───────────────────────────────────────────────────────────
def evaluate(model, name):
    y_pred = model.predict(X_test)
    rmse   = float(np.sqrt(mean_squared_error(y_test, y_pred)))
    mae    = float(mean_absolute_error(y_test, y_pred))
    r2     = float(r2_score(y_test, y_pred))
    nz     = y_test != 0
    mape   = float(np.mean(np.abs((y_test[nz] - y_pred[nz]) / y_test[nz])) * 100)
    step   = max(1, len(y_pred) // 500)
    return {
        'RMSE': round(rmse, 1), 'MAE': round(mae, 1),
        'R2':   round(r2, 4),   'MAPE': round(mape, 1),
        'y_pred': y_pred, 'y_test': np.array(y_test),
        'y_pred_s': y_pred[::step], 'y_test_s': np.array(y_test)[::step],
        'resid_s':  (np.array(y_test) - y_pred)[::step],
    }

results = {name: evaluate(m, name) for name, m in models.items()}

comp = pd.DataFrame([
    {'Model': n, 'RMSE (h)': r['RMSE'], 'MAE (h)': r['MAE'],
     'R²': r['R2'], 'MAPE (%)': r['MAPE']}
    for n, r in results.items()
]).sort_values('RMSE (h)')

print('=== Model comparison — test set ===')
print(comp.to_string(index=False))
print()
print('Metric guide:')
print('  RMSE  — root mean squared error in hours; penalises large errors heavily')
print('  MAE   — mean absolute error in hours; more robust to extreme outliers')
print('  R²    — % of variance explained (1.0 = perfect). Low values are expected')
print('          on this dataset due to extreme right-skew (skewness ~6.4).')
print('  MAPE  — mean absolute % error; high because many requests close in ~0h')

In [ ]:
# ── Metric comparison bar charts ──────────────────────────────────────────
model_names = list(results.keys())
colors = [MODEL_COLORS.get(n, C_BLUE) for n in model_names]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
metric_cfg = [
    ('RMSE',  'RMSE (hours)', False),
    ('MAE',   'MAE (hours)',  False),
    ('R2',    'R²',           True),   # True = higher is better
]

for ax, (key, label, higher_better) in zip(axes, metric_cfg):
    vals  = [results[n][key] for n in model_names]
    bars  = ax.bar(model_names, vals, color=colors, alpha=0.85, edgecolor='none')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + max(vals) * 0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9, color=C_GRAY)
    best_idx = (vals.index(max(vals)) if higher_better
                else vals.index(min(vals)))
    bars[best_idx].set_edgecolor('#2C2C2A')
    bars[best_idx].set_linewidth(1.8)
    ax.set_title(f'{label}\n(best outlined)', fontweight='500', fontsize=10)
    ax.set_xticklabels(model_names, rotation=18, ha='right', fontsize=9)

plt.suptitle('Test-set performance — all models', fontweight='500', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Predicted vs actual (top) and residual plots (bottom) ─────────────────
n_models = len(results)
fig, axes = plt.subplots(2, n_models, figsize=(5 * n_models, 9))
if n_models == 1:
    axes = axes.reshape(2, 1)

for col, (name, r) in enumerate(results.items()):
    color = MODEL_COLORS.get(name, C_BLUE)
    y_pred_s = r['y_pred_s']
    y_test_s = r['y_test_s']
    resid_s  = r['resid_s']

    # ── Top: predicted vs actual ─────────────────────────────────────────
    ax = axes[0, col]
    ax.scatter(y_test_s, y_pred_s, alpha=0.18, s=6,
               color=color, edgecolors='none')
    lim = max(float(y_test_s.max()), float(y_pred_s.max()))
    ax.plot([0, lim], [0, lim], color=C_RED, lw=1.2,
            linestyle='--', label='Perfect')
    ax.set_xlabel('Actual hours to close')
    ax.set_ylabel('Predicted hours to close')
    ax.set_title(
        f'{name}\nR²={r["R2"]:.3f}   RMSE={r["RMSE"]:.0f}h',
        fontweight='500', fontsize=10
    )
    ax.legend(fontsize=8)

    # ── Bottom: residuals vs predicted ───────────────────────────────────
    ax = axes[1, col]
    ax.scatter(y_pred_s, resid_s, alpha=0.18, s=6,
               color=color, edgecolors='none')
    ax.axhline(0, color=C_RED, lw=1.2, linestyle='--')
    ax.set_xlabel('Predicted hours to close')
    ax.set_ylabel('Residual  (actual − predicted)')
    ax.set_title(
        f'Residuals — {name}\nMAE={r["MAE"]:.0f}h',
        fontsize=10
    )

for ax in axes.flat:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('Predicted vs actual (top row)   |   Residuals (bottom row)',
             fontweight='500', y=1.01)
plt.tight_layout()
plt.show()
print('What to look for:')
print('  Top row:    points should hug the red diagonal for a good model.')
print('  Bottom row: residuals should be randomly scattered around 0.')
print('              A fan shape (wider at the right) = heteroscedasticity —')
print('              the model struggles more with long-close requests.')

In [ ]:
# ── Feature importances ───────────────────────────────────────────────────
TOP_N = 12
n_models = len(models)
fig, axes = plt.subplots(1, n_models, figsize=(5 * n_models, 6), sharey=False)
if n_models == 1:
    axes = [axes]

for ax, (name, model) in zip(axes, models.items()):
    imp = (
        pd.Series(model.feature_importances_, index=FEATURE_NAMES)
        .sort_values(ascending=False)
        .head(TOP_N)
    )
    ax.barh(
        imp.index[::-1], imp.values[::-1],
        color=MODEL_COLORS.get(name, C_BLUE),
        alpha=0.85, height=0.7, edgecolor='none'
    )
    ax.set_xlabel('Importance score')
    ax.set_title(name, fontweight='500')
    ax.tick_params(axis='y', labelsize=8)
    for spine in ['top','right']: ax.spines[spine].set_visible(False)

plt.suptitle(f'Top {TOP_N} feature importances by model',
             fontweight='500', y=1.01)
plt.tight_layout()
plt.show()

# Consensus top features across ensemble models
ensemble_names = [n for n in models if n != 'Decision Tree']
if ensemble_names:
    avg_imp = pd.DataFrame({
        n: pd.Series(models[n].feature_importances_, index=FEATURE_NAMES)
        for n in ensemble_names
    }).mean(axis=1).sort_values(ascending=False)
    print(f'Top 10 features by average importance (ensemble models):')
    for feat, val in avg_imp.head(10).items():
        print(f'  {val:.4f}  {feat}')

## Decision Tree: Structure & Interpretability

Unlike ensemble models, a single Decision Tree can be fully read and explained.
We train a shallow version (`max_depth=4`) to keep the diagram readable.

In [ ]:
# ── Shallow tree — interpretable visual ──────────────────────────────────
dt_shallow = DecisionTreeRegressor(
    max_depth=4,
    min_samples_leaf=100,
    random_state=RANDOM_STATE,
)
dt_shallow.fit(X_train, y_train)

shallow_pred  = dt_shallow.predict(X_test)
shallow_rmse  = float(np.sqrt(mean_squared_error(y_test, shallow_pred)))
shallow_r2    = float(r2_score(y_test, shallow_pred))
full_rmse     = results['Decision Tree']['RMSE']
full_r2       = results['Decision Tree']['R2']

print(f'Shallow tree (depth=4):  RMSE={shallow_rmse:.1f}h   R²={shallow_r2:.4f}')
print(f'Full tree   (depth=10):  RMSE={full_rmse:.1f}h   R²={full_r2:.4f}')
print(f'Accuracy cost of interpretability: +{shallow_rmse - full_rmse:.1f}h RMSE')
print(f'Leaves in shallow tree: {dt_shallow.get_n_leaves()}')

fig, ax = plt.subplots(figsize=(24, 10))
plot_tree(
    dt_shallow,
    feature_names=FEATURE_NAMES,
    filled=True,
    rounded=True,
    impurity=False,   # hide MSE — keeps labels clean
    precision=0,      # round values to integers
    fontsize=8,
    ax=ax,
)
ax.set_title(
    f'Decision Tree structure  (max_depth=4, {dt_shallow.get_n_leaves()} leaves)\n'
    'Each node shows: split condition  |  samples  |  predicted hours_to_close',
    fontweight='500', pad=14, fontsize=11
)
plt.tight_layout()
plt.show()

print('\nText representation (useful for reports / copy-paste):')
print(export_text(dt_shallow, feature_names=FEATURE_NAMES, max_depth=3))

In [ ]:
# ── Bias-variance trade-off: depth vs RMSE ────────────────────────────────
depths = [2, 3, 4, 5, 6, 8, 10, 12, 15, None]
rows   = []
for d in depths:
    m = DecisionTreeRegressor(
        max_depth=d, min_samples_leaf=5, random_state=RANDOM_STATE
    )
    m.fit(X_train, y_train)
    train_rmse = float(np.sqrt(mean_squared_error(y_train, m.predict(X_train))))
    test_rmse  = float(np.sqrt(mean_squared_error(y_test,  m.predict(X_test))))
    rows.append({'depth': str(d) if d else 'None\n(full)',
                 'train_rmse': train_rmse, 'test_rmse': test_rmse})
dr = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(10, 4))
x = range(len(dr))
ax.plot(x, dr['train_rmse'], marker='o', color=C_TEAL,
        linewidth=2, label='Train RMSE')
ax.plot(x, dr['test_rmse'],  marker='s', color=C_RED,
        linewidth=2, label='Test RMSE')
ax.set_xticks(list(x))
ax.set_xticklabels(dr['depth'], fontsize=9)
ax.set_xlabel('max_depth')
ax.set_ylabel('RMSE (hours)')
ax.set_title('Bias-variance trade-off: depth vs RMSE',
             fontweight='500')
ax.legend()
best_idx = int(dr['test_rmse'].idxmin())
ax.axvline(best_idx, color=C_GRAY, linewidth=1, linestyle=':')
ax.annotate(
    f'Best test RMSE\n(depth={dr.loc[best_idx, "depth"]})',
    xy=(best_idx, dr.loc[best_idx, 'test_rmse']),
    xytext=(best_idx + 0.5, dr.loc[best_idx, 'test_rmse'] + 30),
    fontsize=9, color=C_GRAY,
    arrowprops=dict(arrowstyle='->', color=C_GRAY, lw=1),
)
plt.tight_layout()
plt.show()
print(dr[['depth','train_rmse','test_rmse']].round(1).to_string(index=False))

In [ ]:
# ── Median absolute error per close-time bucket ───────────────────────────
# Shows WHERE each model struggles — useful for understanding model limits.
bins   = [0, 24, 72, 168, 336, 720, np.inf]
labels = ['0–24h', '24–72h', '72–168h', '168–336h', '336–720h', '>720h']
y_test_arr = np.array(y_test)
bin_col    = pd.cut(y_test_arr, bins=bins, labels=labels)

n_models = len(models)
fig, ax  = plt.subplots(figsize=(12, 5))
x = np.arange(len(labels))
w = 0.8 / n_models

for i, (name, model) in enumerate(models.items()):
    y_pred  = model.predict(X_test)
    abs_err = np.abs(y_test_arr - y_pred)
    med_err = pd.Series(abs_err).groupby(bin_col, observed=True).median()
    ax.bar(
        x + i * w - (n_models - 1) * w / 2,
        med_err.values,
        width=w, label=name,
        color=MODEL_COLORS.get(name, C_BLUE),
        alpha=0.85, edgecolor='none'
    )

ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_xlabel('Actual hours to close (bucket)')
ax.set_ylabel('Median absolute error (hours)')
ax.set_title('Where does each model struggle? Median absolute error by bucket',
             fontweight='500')
ax.legend(fontsize=9)
for spine in ['top','right']: ax.spines[spine].set_visible(False)
plt.tight_layout()
plt.show()

counts = pd.Series(y_test_arr).groupby(bin_col, observed=True).count()
print('Test-set counts per bucket:')
for lbl, cnt in zip(labels, counts):
    print(f'  {lbl:12s}: {cnt:,} requests  ({cnt/len(y_test)*100:.1f}%)')

In [ ]:
# ── Bonus: does log-transforming the target help? ─────────────────────────
# Given the heavy right skew, training on log(1 + hours_to_close) and
# back-transforming predictions often improves RMSE on the original scale.
y_train_log = np.log1p(y_train)
y_test_log  = np.log1p(y_test)

dt_log = DecisionTreeRegressor(
    max_depth=10, min_samples_leaf=5, random_state=RANDOM_STATE
)
dt_log.fit(X_train, y_train_log)
pred_log = np.expm1(dt_log.predict(X_test))   # back-transform

rf_log = RandomForestRegressor(
    n_estimators=200, max_depth=10, min_samples_leaf=5,
    random_state=RANDOM_STATE, n_jobs=-1
)
rf_log.fit(X_train, y_train_log)
pred_rf_log = np.expm1(rf_log.predict(X_test))

print('Effect of log-transforming the target:')
print(f'{"Model":<22}  {"Standard RMSE":>14}  {"Log-target RMSE":>16}  {"Improvement":>12}')
for name, std_pred, log_pred in [
    ('Decision Tree', results['Decision Tree']['y_pred'], pred_log),
    ('Random Forest', results['Random Forest']['y_pred'], pred_rf_log),
]:
    std_rmse = float(np.sqrt(mean_squared_error(y_test, std_pred)))
    log_rmse = float(np.sqrt(mean_squared_error(y_test, log_pred)))
    diff = std_rmse - log_rmse
    sign = '+' if diff > 0 else ''
    print(f'{name:<22}  {std_rmse:>14.1f}  {log_rmse:>16.1f}  {sign}{diff:>+11.1f}h')
print()
print('A positive improvement means log-transforming reduces RMSE.')
print('If the improvement is large, consider using log(1+y) as your target')
print('and back-transforming predictions with expm1().')